In [2]:
### Question 1: Convert Paragraph to Sentences
# Explanation:
# Given a long paragraph, split it into a list of individual sentences.
# A sentence is defined as ending with '.', '!' or '?'.
# 
# Example:
# Input: "Hello world! How are you? I am fine."
# Output: ["Hello world!", "How are you?", "I am fine."]

# BEGIN SOLUTION
def split_paragraph_to_sentences(paragraph):
    import re
    sentences = re.findall(r'[^.!?]+[.!?]', paragraph)
    return [s.strip() for s in sentences]

# END SOLUTION

# BEGIN TEST
paragraph = "Hello world! How are you? I am fine."
sentences = split_paragraph_to_sentences(paragraph)
assert sentences == ["Hello world!", "How are you?", "I am fine."], f"Unexpected: {sentences}"
# END TEST

In [3]:
### Question 2: Vocabulary Construction
# Instructions:
# Implement a function that builds a vocabulary from a list of sentences.
# The function should assign IDs (token IDs) to words based on their frequency (most frequent gets lowest ID of 0, then next most frequent gets 1, etc).
#
# Input:
#   - sentences: A list of strings, where each string is a sentence
#
# Output:
#   - vocab: A dictionary mapping words (strings) to their IDs (integers)
#
# Example:
# Input: ["hello world", "hello there", "world of AI"]
# Count: {"hello": 2, "world": 2, "there": 1, "of": 1, "AI": 1}
# Sorted: ["hello", "world", "there", "of", "AI"]  # Note: order of equal frequency words may vary
# Output: {"hello": 0, "world": 1, "there": 2, "of": 3, "AI": 4}

# BEGIN SOLUTION
def create_vocabulary(sentences):
    word_counts = {}
    
    # Count word frequencies across all sentences
    for sentence in sentences:
        words = sentence.strip().split()
        for word in words:
            word_counts[word] = word_counts.get(word, 0) + 1
    
    # Sort words by frequency (descending)
    sorted_words = sorted(word_counts.keys(), key=lambda w: word_counts[w], reverse=True)
    
    # Create vocabulary with IDs
    vocab = {}
    for i, word in enumerate(sorted_words):
        vocab[word] = i
        
    return vocab

# END SOLUTION

# BEGIN TEST
sentences = ["hello world", "hello there", "world of AI"]
vocab = create_vocabulary(sentences)
assert set(vocab.keys()) == {"hello", "world", "there", "of", "AI"}, f"Unexpected vocab keys: {vocab.keys()}"
assert vocab["hello"] < vocab["world"] or vocab["world"] < vocab["hello"], "Most frequent words should have lowest IDs"
assert all(vocab["hello"] < vocab[w] or vocab["world"] < vocab[w] for w in ["there", "of", "AI"]), "Less frequent words should have higher IDs"
# END TEST

In [4]:
### Question 3: Tokenization
# Instructions:
# Implement a function that converts a sentence into a list of token IDs using a given vocabulary.
# Any word not found in the vocabulary should be assigned the 'unknown_token' ID.
#
# Input:
#   - sentence: A string containing words separated by spaces
#   - vocab: A dictionary mapping words to their unique token IDs (integers)
#   - unknown_token: An integer token ID to use for words not in the vocabulary (default: -1)
#
# Output:
#   - tokens: A list of integers representing the token IDs for each word in the sentence
#
# Example:
# vocab = {"hello": 0, "world": 1}
# Input: "hello world hello new"
# Output: [0, 1, 0, -1]

# BEGIN SOLUTION
def tokenize(sentence, vocab, unknown_token=-1):
    words = sentence.strip().split()
    tokens = []
    for word in words:
        token_id = vocab.get(word, unknown_token)
        tokens.append(token_id)
    return tokens

# END SOLUTION

# BEGIN TEST
vocab = {"hello": 0, "world": 1}
tokens = tokenize("hello world hello", vocab)
assert tokens == [0, 1, 0], f"Unexpected tokens: {tokens}"
tokens_with_unknown = tokenize("hello brave world", vocab)
assert tokens_with_unknown == [0, -1, 1], f"Unknown word handling failed: {tokens_with_unknown}"
# END TEST

In [21]:
### Question 4: Embedding Lookup
# Instructions:
# Implement a function that converts token IDs to their corresponding embedding vectors by indexing into the embedding matrix using the token IDs.
# The function should also handle unknown tokens explicitly by mapping them to a special embedding vector.
#
# Input:
#   - token_ids: A list or array of integers representing token IDs
#   - embedding_matrix: A 2D NumPy array where each row is an embedding vector
#                     Shape: (vocab_size, embedding_dim)
#   - unknown_token: The token ID used for unknown tokens (default: -1)
#   - unknown_embedding: Optional embedding vector for unknown tokens (default: None, which means use a zero vector of appropriate dimension for unknown tokens)
#
# Output:
#   - embeddings: A 2D NumPy array of embedding vectors for the input token IDs
#                Shape: (len(token_ids), embedding_dim)
#
# Example:
# token_ids = [2, 0, 1, -1]  # -1 represents unknown token
# embedding_matrix = [
#   [1.0, 0.0],  # Embedding for token ID 0
#   [0.0, 1.0],  # Embedding for token ID 1
#   [-1.0, 1.0]  # Embedding for token ID 2
# ]
# Output:
#   [[-1.0, 1.0],  # Embedding for token ID 2
#    [1.0, 0.0],   # Embedding for token ID 0
#    [0.0, 1.0],   # Embedding for token ID 1
#    [0.0, 0.0]]   # Embedding for unknown token (zeros by default)

# BEGIN SOLUTION
import numpy as np

def embedding_lookup(token_ids, embedding_matrix, unknown_token=-1, unknown_embedding=None):
    """
    Convert token IDs to their corresponding embedding vectors, with explicit handling for unknown tokens.

    Args:
        token_ids: A list or array of integers representing token IDs
        embedding_matrix: A 2D NumPy array where each row is an embedding vector
        unknown_token: The ID used for unknown tokens (default: -1)
        unknown_embedding: Optional embedding vector for unknown tokens.
                          If None, uses a zero vector of appropriate dimension
    
    Returns:
        A 2D NumPy array of embedding vectors for the input token IDs
    """
    # Convert token_ids to numpy array if it's not already
    token_ids = np.array(token_ids)
    
    # If unknown_embedding is not provided, create a zero vector
    if unknown_embedding is None:
        unknown_embedding = np.zeros(embedding_matrix.shape[1], dtype=np.float64)
    else:
        unknown_embedding = np.array(unknown_embedding, dtype=np.float64)  # Ensure it's a numpy array with float dtype
    
    # Create the result array
    result = np.zeros((len(token_ids), embedding_matrix.shape[1]), dtype=np.float64)
    
    # Handle known tokens
    known_mask = (token_ids != unknown_token)
    if np.any(known_mask):
        result[known_mask] = embedding_matrix[token_ids[known_mask]].astype(np.float64)
    
    # Handle unknown tokens
    unknown_mask = (token_ids == unknown_token)
    if np.any(unknown_mask):
        result[unknown_mask] = unknown_embedding
    
    return result

# END SOLUTION

# BEGIN TEST
E = np.array([[1, 0], [0, 1], [-1, 1]])
tokens = [2, 0, 1]
output = embedding_lookup(tokens, E)
expected = np.array([[-1, 1], [1, 0], [0, 1]])
assert np.array_equal(output, expected), f"Unexpected output: {output}"

# Test with unknown tokens
tokens_with_unknown = [2, -1, 1]
output_with_unknown = embedding_lookup(tokens_with_unknown, E)
# Unknown token should be mapped to zeros by default
expected_with_unknown = np.array([[-1, 1], [0, 0], [0, 1]])
assert np.array_equal(output_with_unknown, expected_with_unknown), f"Unknown token handling failed: {output_with_unknown}"

# Test with custom unknown embedding
custom_unknown = np.array([0.5, 0.5])
output_with_custom = embedding_lookup(tokens_with_unknown, E, unknown_embedding=custom_unknown)
expected_with_custom = np.array([[-1, 1], [0.5, 0.5], [0, 1]])
assert np.array_equal(output_with_custom, expected_with_custom), f"Custom unknown embedding failed: {output_with_custom}"
# END TEST

In [22]:
### Question 5: Input/Target Dataset Preparation For Training
# Instructions:
# Implement a function that creates input-target pairs for language model training.
# Each input will be a subsequence of token IDs of length 'context_size', 
# and each target will be the next token after that subsequence.
#
# Input:
#   - tokens: A list of token IDs representing a tokenized text sequence
#   - context_size: An integer specifying how many tokens to use as context (input)
#
# Output:
#   - pairs: A list of tuples, where each tuple contains:
#            (input_sequence, target_token)
#            input_sequence is a list of context_size token IDs
#            target_token is a single token ID that follows the input_sequence
#
# Example:
# tokens = [5, 2, 7, 3, 1]
# context_size = 2
# Process:
#   - First pair: Input=[5, 2], Target=7
#   - Second pair: Input=[2, 7], Target=3
#   - Third pair: Input=[7, 3], Target=1
# Output: [([5, 2], 7), ([2, 7], 3), ([7, 3], 1)]

# BEGIN SOLUTION
def create_training_pairs(tokens, context_size):
    pairs = []
    for i in range(len(tokens) - context_size):
        x = tokens[i:i + context_size]
        y = tokens[i + context_size]
        pairs.append((x, y))
    return pairs

# END SOLUTION

# BEGIN TEST
context = 2
toks = [0, 1, 2, 3]
result = create_training_pairs(toks, context)
assert result == [([0, 1], 2), ([1, 2], 3)], f"Unexpected result: {result}"
# END TEST

In [7]:
### Question 6: Masked Self-Attention (Single Head)
# Instructions:
# Implement a function that computes masked self-attention for a single attention head.
# The function will take as input the raw input embeddings and weight matrices for Q, K, V.
# The mask prevents tokens from attending to future positions (causal/autoregressive attention).
#
# More details about the mathematical computations for this question:
# 1. Compute Q, K, V matrices from input X and the weight matrices for Q, K, V
# 2. Compute attention scores: scores = Q × K^T / sqrt(d_k)
#    where d_k is the dimension of the key vectors
# 3. Apply causal mask: use -1e9 in positions that should not be attended to
# 4. Apply softmax with numerical stability: softmax(x) = exp(x - max(x)) / sum(exp(x - max(x)))
# 5. Compute the final output by multiplying the attention weights with the value matrix V
#
# Input:
#   - X: Input embeddings of shape (seq_length, d_model)
#   - W_Q: Query weight matrix of shape (d_model, d_k)
#   - W_K: Key weight matrix of shape (d_model, d_k)
#   - W_V: Value weight matrix of shape (d_model, d_v)
#
# Output:
#   - output: Attention output of shape (seq_length, d_v)

# BEGIN SOLUTION
def masked_attention(X, W_Q, W_K, W_V):
    # Compute Q, K, V matrices
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V
    
    # Compute attention scores
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    
    # Create and apply causal mask
    mask = np.triu(np.ones_like(scores), k=1) * -1e9
    masked_scores = scores + mask
    
    # Apply softmax to get attention weights
    weights = np.exp(masked_scores - np.max(masked_scores, axis=-1, keepdims=True))
    weights = weights / weights.sum(axis=-1, keepdims=True)
    
    # Compute output
    return weights @ V

# END SOLUTION

# BEGIN TEST
# Small test case
X = np.array([[1.0, 0.0], [0.0, 1.0]])
W_Q = np.array([[1.0, 0.0], [0.0, 1.0]])
W_K = np.array([[1.0, 0.0], [0.0, 1.0]])
W_V = np.array([[1.0, 2.0], [3.0, 4.0]])
out = masked_attention(X, W_Q, W_K, W_V)

assert out.shape == (2, 2), f"Unexpected shape: {out.shape}"
# Check that the first token only attends to itself
expected_first_token = X[0] @ W_V
assert np.allclose(out[0], expected_first_token), "First token should only attend to itself"
# Check that the second token can attend to both first and second tokens
assert not np.allclose(out[1], X[1] @ W_V), "Second token should attend to both positions"

# More complex test
X = np.random.randn(3, 4)  # 3 tokens, d_model=4
W_Q = np.random.randn(4, 2)  # d_model=4, d_k=2
W_K = np.random.randn(4, 2)  # d_model=4, d_k=2
W_V = np.random.randn(4, 3)  # d_model=4, d_v=3
out = masked_attention(X, W_Q, W_K, W_V)
assert out.shape == (3, 3), f"Unexpected shape for larger test: {out.shape}"

# Explicitly compute Q, K, V for comparison
Q = X @ W_Q
K = X @ W_K
V = X @ W_V
# Check that first token only attends to itself
manual_first_output = (np.array([1.0, 0.0, 0.0]) @ V)
assert np.allclose(out[0], manual_first_output), "First token should only attend to itself in complex test"
# END TEST

In [8]:
### Question 7: Layer Normalization
# Instructions:
# Implement Layer Normalization as used in LLM/GPT architectures. In transformers, Layer Normalization
# normalizes the hidden representation of each token position independently.
# This stabilizes training by ensuring that the hidden state values have consistent scale throughout
# the network. After normalization, the values are scaled and shifted using learnable parameters gamma and beta.
#
# More Details about the mathematical computations for this question:
# For each token position in the sequence:
# 1. Compute mean: μ = (1/d_model) ∑ xᵢ
# 2. Compute variance: σ² = (1/d_model) ∑ (xᵢ - μ)² 
# 3. Normalize: x̂ = (x - μ) / sqrt(σ² + ε)
# 4. Scale and shift: y = γ · x̂ + β
#
# Input:
#   - x: Hidden representation tensor of shape (seq_length, d_model)
#   - gamma: Scale parameter of shape (d_model,)
#   - beta: Shift parameter of shape (d_model,)
#   - eps: Small constant for numerical stability (default: 1e-5)
#
# Output:
#   - normalized: Normalized tensors with the same shape as input

# BEGIN SOLUTION
def layer_norm(x, gamma, beta, eps=1e-5):
    """
    Apply Layer Normalization to the hidden states.
    
    Args:
        x: Hidden tensor of shape (seq_length, d_model)
        gamma: Scale parameter of shape (d_model,)
        beta: Shift parameter of shape (d_model,)
        eps: Small constant for numerical stability
        
    Returns:
        Normalized hidden tensor of shape (seq_length, d_model)
    """
    # Calculate mean and variance along the hidden dimension (last axis)
    mean = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    
    # Normalize
    norm_x = (x - mean) / np.sqrt(var + eps)
    
    # Scale and shift
    return gamma * norm_x + beta

# END SOLUTION

# BEGIN TEST
x = np.array([[1.0, 2.0], [3.0, 4.0]])
gamma = np.ones(2)
beta = np.zeros(2)
output = layer_norm(x, gamma, beta)

# Test that mean is close to 0 and std is close to 1 along the hidden dimension
assert np.allclose(output.mean(axis=-1), 0, atol=1e-3), "Mean along hidden dimension should be close to 0"
assert np.allclose(output.std(axis=-1), 1, atol=1e-3), "Standard deviation along hidden dimension should be close to 1"

# Test invariance to input scale (important property of Layer Norm in transformers)
scaled_x = x * 10
scaled_output = layer_norm(scaled_x, gamma, beta)
assert np.allclose(output, scaled_output, atol=1e-3), "Layer norm should be invariant to input scale"
# END TEST

In [ ]:
### Question 8: Feedforward Network
# Instructions:
# Implement a two-layer feedforward neural network with a ReLU activation function
# used in transformer blocks. The input must be a 3D tensor of shape (batch_size, seq_length, d_model).
#
# Mathematical Representation:
# 1. First layer: h = ReLU(x × W₁ + b₁)
#    where ReLU(z) = max(0, z) is applied element-wise
# 2. Second layer: output = h × W₂ + b₂
#
# The feedforward network is applied to each token position independently but identically.
#
# Input:
#   - x: Input tensor of shape (batch_size, seq_length, d_model)
#   - W1: Weight matrix for first layer of shape (d_model, d_ff)
#   - b1: Bias vector for first layer of shape (d_ff,)
#   - W2: Weight matrix for second layer of shape (d_ff, d_model)
#   - b2: Bias vector for second layer of shape (d_model,)
#
# Output:
#   - output: Output tensor of shape (batch_size, seq_length, d_model)
#

# BEGIN SOLUTION
def feedforward(x, W1, b1, W2, b2):
    """
    Apply a feedforward neural network to each position in the input sequence.
    
    Args:
        x: Input tensor of shape (batch_size, seq_length, d_model)
        W1: Weight matrix for first layer of shape (d_model, d_ff)
        b1: Bias vector for first layer of shape (d_ff,)
        W2: Weight matrix for second layer of shape (d_ff, d_model)
        b2: Bias vector for second layer of shape (d_model,)
        
    Returns:
        Output tensor of shape (batch_size, seq_length, d_model)
    """
    # Ensure x is 3D
    if len(x.shape) != 3:
        raise ValueError(f"Input must be 3D tensor with shape (batch_size, seq_length, d_model), got shape {x.shape}")
    
    # Get dimensions
    batch_size, seq_length, d_model = x.shape
    print(x.shape)
    
    # Reshape to 2D for batch matrix multiplication
    x_reshaped = x.reshape(-1, d_model)
    print(x_reshaped.shape)
    
    # First layer with ReLU activation
    h = np.maximum(0, x_reshaped @ W1 + b1)
    
    # Second layer
    output = h @ W2 + b2
    
    # Reshape back to 3D
    return output.reshape(batch_size, seq_length, -1)

# END SOLUTION

# BEGIN TEST
# Test with 3D input
batch_size, seq_length, d_model, d_ff = 2, 3, 4, 8
x = np.random.randn(batch_size, seq_length, d_model)
W1 = np.random.randn(d_model, d_ff)
b1 = np.random.randn(d_ff)
W2 = np.random.randn(d_ff, d_model)
b2 = np.random.randn(d_model)

output = feedforward(x, W1, b1, W2, b2)

# Check output shape
assert output.shape == (batch_size, seq_length, d_model), f"Expected shape {(batch_size, seq_length, d_model)}, got {output.shape}"

# Test with a simple example
x_simple = np.array([[[1.0, 2.0]]])  # batch_size=1, seq_length=1, d_model=2
W1_simple = np.array([[1.0, 0.0], [0.0, 1.0]])
b1_simple = np.zeros(2)
W2_simple = np.array([[1.0, 0.0], [0.0, 1.0]])
b2_simple = np.zeros(2)
output_simple = feedforward(x_simple, W1_simple, b1_simple, W2_simple, b2_simple)
expected_simple = np.array([[[1.0, 2.0]]])
assert np.allclose(output_simple, expected_simple), f"Expected {expected_simple}, got {output_simple}"

# Test ReLU functionality with negative values
x_neg = np.array([[[-1.0, 2.0]]])
output_neg = feedforward(x_neg, W1_simple, b1_simple, W2_simple, b2_simple)
expected_neg = np.array([[[0.0, 2.0]]])
assert np.allclose(output_neg, expected_neg), f"ReLU not working correctly. Expected {expected_neg}, got {output_neg}"

# Test error handling for incorrect input shape
try:
    x_wrong = np.array([[1.0, 2.0]])  # 2D instead of 3D
    feedforward(x_wrong, W1_simple, b1_simple, W2_simple, b2_simple)
    assert False, "Should have raised ValueError for incorrect input shape"
except ValueError:
    pass  # Expected behavior
# END TEST

(2, 3, 4)
(6, 4)
(1, 1, 2)
(1, 2)
(1, 1, 2)
(1, 2)


In [10]:
### Question 9: Skip Connection + Dropout
# Instructions:
# Implement a function that applies a skip connection followed by dropout.
# The skip connection adds the input to the output of the previous layer (ffn_output),
# and dropout randomly sets elements to zero with probability dropout_rate.
#
# Mathematical Representation:
# 1. Skip Connection: z = x + ffn_output
# 2. Dropout: output = z * mask
#    where mask is a tensor of 0s and 1s with P(mask_ij = 0) = dropout_rate
#
# Input:
#   - x: Original input tensor of any shape
#   - ffn_output: Output from the previous layer, same shape as x
#   - dropout_rate: Probability of setting an element to zero (default: 0.5)
#   - seed: Random seed for reproducibility (default: 42)
#
# Output:
#   - output: Tensor after skip connection and dropout, same shape as x

# BEGIN SOLUTION
def residual_dropout(x, ffn_output, dropout_rate=0.5, seed=42):
    np.random.seed(seed)
    mask = np.random.rand(*x.shape) > dropout_rate
    return (x + ffn_output) * mask

# END SOLUTION

# BEGIN TEST
x = np.ones((2, 2))
y = np.ones((2, 2))
# Test with no dropout
out = residual_dropout(x, y, dropout_rate=0.0)
assert np.array_equal(out, x + y), f"Without dropout, output should equal x + y"

# Test with 100% dropout
out_full_drop = residual_dropout(x, y, dropout_rate=1.0)
assert np.array_equal(out_full_drop, np.zeros_like(x)), f"With 100% dropout, output should be all zeros"

# Test with fixed seed for reproducibility
np.random.seed(42)
ref_mask = np.random.rand(2, 2) > 0.5
np.random.seed(42)
out_partial = residual_dropout(x, y, dropout_rate=0.5, seed=42)
assert np.array_equal(out_partial, (x + y) * ref_mask), f"Dropout mask not applied correctly"
# END TEST

In [11]:
### Question 10: Linear Output Projection
# Instructions:
# Implement a function that projects the transformer's hidden states to logits
# for vocabulary prediction using a linear layer.
#
# Mathematical Representation:
# output = hidden_states × W_out + b_out
#
# Input:
#   - hidden_states: Transformer output tensor of shape (batch_size, seq_len, d_model)
#   - W_out: Weight matrix of shape (d_model, vocab_size)
#   - b_out: Bias vector of shape (vocab_size,)
#
# Output:
#   - logits: Unnormalized prediction scores of shape (batch_size, seq_len, vocab_size)

# BEGIN SOLUTION
def output_projection(hidden_states, W_out, b_out):
    """
    Project transformer outputs to vocabulary-sized logits.
    
    Args:
        hidden_states: Transformer output of shape (batch_size, seq_len, d_model)
        W_out: Output projection matrix of shape (d_model, vocab_size)
        b_out: Output bias of shape (vocab_size,)
        
    Returns:
        Logits of shape (batch_size, seq_len, vocab_size)
    """
    return hidden_states @ W_out + b_out

# END SOLUTION

# BEGIN TEST
batch_size, seq_len, d_model = 2, 3, 4
vocab_size = 5
hidden_states = np.random.rand(batch_size, seq_len, d_model)
W_out = np.random.rand(d_model, vocab_size)
b_out = np.random.rand(vocab_size)

logits = output_projection(hidden_states, W_out, b_out)

# Check shape
assert logits.shape == (batch_size, seq_len, vocab_size), f"Expected shape {(batch_size, seq_len, vocab_size)}, got {logits.shape}"

# Check values (manual calculation for first position)
expected_first = hidden_states[0, 0] @ W_out + b_out
assert np.allclose(logits[0, 0], expected_first), "Output projection calculation is incorrect"
# END TEST


In [12]:
### Question 11: Temperature Scaling
# Instructions:
# Implement temperature scaling for controlling the randomness of the output distribution.
# Higher temperature produces a more uniform distribution, while lower temperature
# makes the distribution more peaked (concentrated on the most likely tokens).
#
# Mathematical Representation:
# temperature_softmax(z) = softmax(z / temperature)
#
# Input:
#   - logits: Unnormalized prediction scores of [vocab_size]
#   - temperature: Positive scalar controlling the smoothness of the distribution
#                 (default: 1.0, which is standard softmax)
#
# Output:
#   - probabilities: Temperature-scaled probability distribution of the same shape as logits
#

# BEGIN SOLUTION
def temperature_scaled_softmax(logits, temperature):
    scaled = logits / temperature
    weights = np.exp(scaled - np.max(scaled, axis=-1, keepdims=True))
    weights = weights / weights.sum(axis=-1, keepdims=True)
    return weights

# END SOLUTION

# BEGIN TEST
logits = np.array([[1.0, 2.0, 3.0]])

# Test with standard temperature (1.0)
std_probs = temperature_scaled_softmax(logits, temperature=1.0)
# assert np.allclose(std_probs, softmax(logits)), "Temperature=1.0 should be equivalent to standard softmax"

# Test with lower temperature (0.5) - should be more peaked
low_temp_probs = temperature_scaled_softmax(logits, temperature=0.5)
assert low_temp_probs[0, 2] > std_probs[0, 2], "Lower temperature should increase probability of highest logit"

# Test with higher temperature (2.0) - should be more uniform
high_temp_probs = temperature_scaled_softmax(logits, temperature=2.0)
assert high_temp_probs[0, 2] < std_probs[0, 2], "Higher temperature should decrease probability of highest logit"
assert high_temp_probs[0, 0] > std_probs[0, 0], "Higher temperature should increase probability of lowest logit"
# END TEST


In [ ]:
### Question 12: Top-k Sampling
# Instructions:
# Implement top-k sampling, a technique for controlling the randomness of text generation
# by limiting sampling to only the k most likely next tokens.
#
# Steps:
# 1. Find the k tokens with the highest logits/probabilities
# 2. Keep only those k tokens and their probabilities
# 3. Renormalize the probabilities to sum to 1
#
# Input:
#   - logits: Unnormalized prediction scores for the next token (shape: [vocab_size])
#   - k: Number of top tokens to consider (integer, k <= vocab_size)
#
# Output:
#   - top_k_indices: Indices of the k tokens with the highest logits (shape: [k])
#   - top_k_probs: Normalized probabilities for the top k tokens (shape: [k])

# BEGIN SOLUTION
def top_k_sampling(logits, k):
    top_k_indices = np.argsort(logits)[-k:][::-1]
    top_k_logits = logits[top_k_indices]
    weights = np.exp(top_k_logits - np.max(top_k_logits, axis=-1, keepdims=True))
    weights = weights / weights.sum(axis=-1, keepdims=True)
    return top_k_indices, weights

# END SOLUTION

# BEGIN TEST
logits = np.array([1.0, 3.0, 2.0, 0.0, -1.0])
k = 2
indices, probs = top_k_sampling(logits, k)

# Test output shape
assert len(indices) == k, f"Expected {k} indices, got {len(indices)}"
assert len(probs) == k, f"Expected {k} probabilities, got {len(probs)}"

# Test that we get the highest logits
assert set(indices) == {1, 2}, f"Expected indices 1 and 2 for top 2 logits, got {indices}"

# Test that probabilities sum to 1
assert np.allclose(np.sum(probs), 1.0), f"Probabilities should sum to 1, got {np.sum(probs)}"

# Test that highest logit has highest probability
assert indices[0] == 1, f"Index with highest logit should be first, got {indices[0]}"
assert probs[0] > probs[1], f"Highest logit should have highest probability"
# END TEST

In [18]:
print("hello world!")

hello world!


In [14]:
### Question 13: Top-p Sampling (Nucleus Sampling)
# Instructions:
# Implement top-p (nucleus) sampling, which selects the smallest set of tokens
# whose cumulative probability exceeds threshold p.
#
# Steps:
# 1. Sort tokens by their probabilities in descending order
# 2. Keep adding tokens to the set until their cumulative probability >= p
# 3. Renormalize the probabilities of the selected tokens to sum to 1
#
# Input:
#   - logits: Unnormalized prediction scores for the next token (shape: [vocab_size])
#   - p: Probability threshold (float between 0 and 1)
#
# Output:
#   - selected_indices: Indices of the selected tokens
#   - selected_probs: Renormalized probabilities for the selected tokens

# BEGIN SOLUTION
def top_p_sampling(logits, p):
    sorted_indices = np.argsort(logits)[::-1]
    sorted_logits = logits[sorted_indices]
    weights = np.exp(sorted_logits - np.max(sorted_logits, axis=-1, keepdims=True))
    weights = weights / weights.sum(axis=-1, keepdims=True)
    cum_probs = np.cumsum(weights)
    cutoff = np.searchsorted(cum_probs, p) + 1
    selected_weights = weights[:cutoff]
    # Renormalize the selected probabilities to sum to 1
    selected_weights = selected_weights / selected_weights.sum()
    return sorted_indices[:cutoff], selected_weights

# END SOLUTION

# BEGIN TEST
logits = np.array([1.0, 3.0, 2.0, 0.0, -1.0])
p = 0.9
indices, norm_probs = top_p_sampling(logits, p)

# Test that probabilities sum to 1
assert np.allclose(np.sum(norm_probs), 1.0), f"Normalized probabilities should sum to 1"

# Test that the sum of original probabilities meets threshold
# original_probs = softmax(logits)[indices]
# assert np.sum(original_probs) >= p, f"Sum of selected probabilities should be at least {p}"

# Test with p=1.0 (should include all tokens)
all_indices, all_probs = top_p_sampling(logits, p=1.0)
assert len(all_indices) == len(logits), f"With p=1.0, all tokens should be included"

# Test with very small p (should include only the most likely token)
small_indices, small_probs = top_p_sampling(logits, p=0.1)
assert len(small_indices) >= 1, f"Should include at least one token"
assert small_indices[0] == np.argmax(logits), f"Most likely token should be included"
# END TEST